# Module 04 — ML Theory & Evaluation
## 04-06: Learning Theory — VC Dimension, PAC Learning & Sample Complexity

**Objective:** Formalise *why* ML models generalise through empirical risk minimisation, Hoeffding's
inequality, VC dimension and shattering, and PAC learning — then empirically validate these theoretical
predictions against real experiments.

**Prerequisites:** 1-07 (Probability & Statistics), 2-01 (Linear Regression), 2-02 (Logistic Regression)

## Part 0 — Setup & Prerequisites

Every ML practitioner knows that models trained on more data generalise better, and that more complex
models require more data.  But *why* is this true, and *how much* data is needed?  This notebook answers
both questions through the lens of **PAC (Probably Approximately Correct) learning theory** — the
mathematical framework that formalises generalisation guarantees for learning algorithms.

The central chain of reasoning is:
1. We want to minimise *true risk* $R(h)$ (error on unseen data), but we only observe *empirical risk* $\hat{R}(h)$ (training error).
2. **Hoeffding's inequality** bounds how far $\hat{R}(h)$ can deviate from $R(h)$ for a *single* hypothesis.
3. **Uniform convergence** extends this to an entire hypothesis class $\mathcal{H}$, at the cost of a $|\mathcal{H}|$ factor.
4. **VC dimension** replaces $|\mathcal{H}|$ with a combinatorial measure of the class's expressive power.
5. **Sample complexity bounds** answer: how many samples suffice to learn with accuracy $\varepsilon$ and confidence $1-\delta$?

> **Note:** This is a theory notebook. Every theorem is validated with runnable code and real data — not lecture notes.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import math
import random
import warnings
import itertools
from typing import Any, Callable
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
N_FEATURES     = 2    # 2-D throughout for visualisation
N_REPEATS      = 30   # repetitions for each (n, complexity) cell to estimate variance
SAMPLE_SIZES   = [20, 50, 100, 200, 400, 800, 1600, 3200]

# Canonical epsilon/delta for PAC bounds
EPSILON = 0.05   # acceptable error: 5% above Bayes error
DELTA   = 0.05   # failure probability: 5%

---
## Part 1 — Theoretical Foundations & Implementations

### 1.1 Empirical Risk Minimisation (ERM)

**True risk** (also called generalisation error or population risk) of a hypothesis $h$ is:
$$R(h) = \mathbb{E}_{(x,y) \sim \mathcal{D}}[\ell(h(x), y)]$$
where $\mathcal{D}$ is the data distribution and $\ell$ is the loss function.
Since $\mathcal{D}$ is unknown, we minimise the **empirical risk** (training error):
$$\hat{R}(h) = \frac{1}{n} \sum_{i=1}^{n} \ell(h(x_i), y_i)$$

The fundamental question: when does $\hat{R}(h) \approx R(h)$?
The answer depends on the **hypothesis class** $\mathcal{H}$ and the **number of samples** $n$.

In [ ]:
# ── Empirical Risk Minimisation ───────────────────────────────────────────────

def zero_one_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    '''Compute the 0-1 loss (fraction of misclassified examples).

    Args:
        y_pred: Predicted class labels of shape (n_samples,).
        y_true: Ground-truth labels of shape (n_samples,).

    Returns:
        Empirical 0-1 risk in [0, 1].
    '''
    return float(np.mean(y_pred != y_true))


def empirical_risk(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
) -> float:
    '''Compute empirical 0-1 risk of a fitted sklearn estimator.

    Args:
        estimator: Fitted classifier with a predict() method.
        X: Feature matrix of shape (n_samples, n_features).
        y: Label vector of shape (n_samples,).

    Returns:
        Empirical risk (error rate) in [0, 1].
    '''
    return zero_one_loss(estimator.predict(X), y)


def generalisation_gap(
    estimator: Any,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> tuple[float, float, float]:
    '''Compute training error, test error, and generalisation gap.

    The generalisation gap is |R_hat(h) - R(h)|, approximating R(h) with
    the test error on a held-out set.  The gap should shrink as n grows.

    Args:
        estimator: Fitted sklearn classifier.
        X_train: Training features.
        y_train: Training labels.
        X_test: Test features.
        y_test: Test labels.

    Returns:
        Tuple of (train_error, test_error, gap).
    '''
    train_err = empirical_risk(estimator, X_train, y_train)
    test_err  = empirical_risk(estimator, X_test, y_test)
    gap       = abs(train_err - test_err)
    return train_err, test_err, gap


# Quick demo: ERM with a linear model and a deep tree

X_demo, y_demo = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=SEED
)
X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_demo, y_demo, test_size=0.33, random_state=SEED
)

for name, clf in [
    ("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
    ("DecisionTree(depth=3)", DecisionTreeClassifier(max_depth=3, random_state=SEED)),
    ("DecisionTree(depth=None)", DecisionTreeClassifier(max_depth=None, random_state=SEED)),
]:
    clf.fit(X_d_tr, y_d_tr)
    tr_err, te_err, gap = generalisation_gap(clf, X_d_tr, y_d_tr, X_d_te, y_d_te)
    print(f"{name:30s}  train={tr_err:.3f}  test={te_err:.3f}  gap={gap:.3f}")

print("\nThe deep tree memorises training data (train≈0) but has a large gap.")

### 1.2 Hoeffding's Inequality & Uniform Convergence

**Hoeffding's inequality** bounds the probability that a sample mean deviates far from its expectation.
For $n$ i.i.d. random variables $Z_1, \ldots, Z_n \in [0,1]$:
$$P\left(\left|\frac{1}{n}\sum_{i=1}^n Z_i - \mathbb{E}[Z]\right| \geq \varepsilon\right) \leq 2\exp(-2n\varepsilon^2)$$

Applied to a **single** hypothesis $h$, with $Z_i = \ell(h(x_i), y_i)$:
$$P\left(|\hat{R}(h) - R(h)| \geq \varepsilon\right) \leq 2e^{-2n\varepsilon^2}$$

**Uniform convergence** (union bound over all $|\mathcal{H}|$ hypotheses):
$$P\left(\sup_{h \in \mathcal{H}} |\hat{R}(h) - R(h)| \geq \varepsilon\right) \leq 2|\mathcal{H}| \cdot e^{-2n\varepsilon^2}$$

Rearranging to a confidence statement: with probability at least $1 - \delta$, for all $h \in \mathcal{H}$:
$$R(h) \leq \hat{R}(h) + \sqrt{\frac{\ln|\mathcal{H}| + \ln(2/\delta)}{2n}}$$

This is the **generalisation bound** — empirical risk plus a complexity penalty.

In [ ]:
# ── Hoeffding's Inequality ────────────────────────────────────────────────────

def hoeffding_bound(
    n: int,
    epsilon: float,
) -> float:
    '''Compute the Hoeffding upper bound on P(|R_hat - R| >= epsilon) for one hypothesis.

    Args:
        n: Number of i.i.d. training samples.
        epsilon: Deviation tolerance.

    Returns:
        Probability upper bound in [0, 1] (may exceed 1 for tiny n).
    '''
    return float(2.0 * math.exp(-2.0 * n * epsilon ** 2))


def uniform_convergence_bound(
    n: int,
    epsilon: float,
    hypothesis_class_size: int,
) -> float:
    '''Compute the uniform convergence bound for a finite hypothesis class.

    Uses the union bound: P(sup_h |R_hat - R| >= eps) <= 2|H|*exp(-2n*eps^2).

    Args:
        n: Number of training samples.
        epsilon: Deviation tolerance.
        hypothesis_class_size: |H|, the number of hypotheses in the class.

    Returns:
        Probability upper bound (may exceed 1 for small n).
    '''
    return float(min(1.0, 2.0 * hypothesis_class_size * math.exp(-2.0 * n * epsilon ** 2)))


def generalisation_bound_finite(
    n: int,
    delta: float,
    hypothesis_class_size: int,
) -> float:
    '''Compute the generalisation bound complexity term for a finite hypothesis class.

    With probability >= 1-delta, for all h: R(h) <= R_hat(h) + complexity_term.

    Args:
        n: Number of training samples.
        delta: Confidence parameter; bound holds with probability 1-delta.
        hypothesis_class_size: |H|.

    Returns:
        The sqrt(…) complexity penalty added to empirical risk.
    '''
    log_term = math.log(hypothesis_class_size) + math.log(2.0 / delta)
    return float(math.sqrt(log_term / (2.0 * n)))


# ── Verify: Hoeffding bound vs empirical deviation over N_REPEATS experiments ─
n_verify = 200
eps_verify = 0.1
n_trials = 5000

# Simulate coin flip: Z_i ~ Bernoulli(0.3), mean = 0.3
true_mean = 0.3
rng_h = np.random.default_rng(SEED)
samples_matrix = rng_h.binomial(1, true_mean, size=(n_trials, n_verify))
sample_means = samples_matrix.mean(axis=1)
empirical_prob = float(np.mean(np.abs(sample_means - true_mean) >= eps_verify))
theoretical_bound = hoeffding_bound(n_verify, eps_verify)

print(f"Hoeffding Inequality Verification (n={n_verify}, eps={eps_verify}):")
print(f"  Empirical P(|mean - E[Z]| >= eps) : {empirical_prob:.4f}")
print(f"  Hoeffding upper bound             : {theoretical_bound:.4f}")
assert empirical_prob <= theoretical_bound + 1e-6, "Hoeffding bound violated!"
print(f"  Bound holds: {empirical_prob:.4f} <= {theoretical_bound:.4f} ✓")

### 1.3 VC Dimension and Shattering

The Hoeffding union bound requires $|\mathcal{H}|$ to be finite.  For real hypothesis classes (e.g., all
linear classifiers in $\mathbb{R}^d$), $|\mathcal{H}| = \infty$.  **VC dimension** replaces $|\mathcal{H}|$
with a finite combinatorial measure.

**Shattering:** A hypothesis class $\mathcal{H}$ *shatters* a set of points $C = \{x_1, \ldots, x_m\}$ if
for every possible labelling $y \in \{0,1\}^m$ there exists $h \in \mathcal{H}$ consistent with that labelling:
$$\forall y \in \{0,1\}^m, \exists h \in \mathcal{H}: h(x_i) = y_i \text{ for all } i$$

**VC dimension:** $\text{VCdim}(\mathcal{H}) = \max\{m : \exists C \text{ of size } m \text{ shattered by } \mathcal{H}\}$

**Key facts:**
- Linear classifiers (hyperplanes) in $\mathbb{R}^d$: $\text{VCdim} = d+1$
- Decision trees with depth $k$: $\text{VCdim} \leq 2^k \log_2(2^k)$
- $k$-NN with $k=1$: $\text{VCdim} = \infty$

In [ ]:
# ── VC Dimension: Shattering Check ────────────────────────────────────────────

def is_shattered_by_linear(
    points: np.ndarray,
    tolerance: float = 1e-6,
) -> bool:
    '''Check whether a linear classifier shatters the given 2-D point set.

    For a set of m points, enumerates all 2^m binary labellings and checks
    whether a linear SVM (hard-margin) can realise each one.  Returns True
    only if all 2^m labellings are realisable.

    Args:
        points: Array of shape (m, 2) — the candidate point set.
        tolerance: Numerical tolerance for SVM convergence check.

    Returns:
        True if the linear class shatters ``points``, False otherwise.
    '''
    m = len(points)
    for labelling in itertools.product([0, 1], repeat=m):
        labels = np.array(labelling)
        # Skip degenerate all-same-class labellings for SVM
        if labels.sum() == 0 or labels.sum() == m:
            continue
        svm = SVC(kernel="linear", C=1e6)  # large C ≈ hard margin
        svm.fit(points, labels)
        y_pred = svm.predict(points)
        if not np.array_equal(y_pred, labels):
            return False
    return True


def empirical_vc_dimension_2d(
    max_points: int = 6,
    n_random_configs: int = 100,
    seed: int = SEED,
) -> int:
    '''Estimate the VC dimension of linear classifiers in 2-D experimentally.

    For each m from 2 to max_points, randomly samples n_random_configs sets
    of m points and checks if any is shattered.  Returns the largest m for
    which at least one configuration is shattered.

    Args:
        max_points: Maximum number of points to try.
        n_random_configs: Number of random point configurations per m.
        seed: Random seed.

    Returns:
        Estimated VC dimension.
    '''
    rng_vc = np.random.default_rng(seed)
    vc_dim = 0
    for m in range(2, max_points + 1):
        found_shattered = False
        for _ in range(n_random_configs):
            pts = rng_vc.standard_normal((m, 2))
            if is_shattered_by_linear(pts):
                found_shattered = True
                break
        if found_shattered:
            vc_dim = m
        else:
            break  # once we fail to shatter m, VCdim < m
    return vc_dim


# Expected: VCdim(linear, R^2) = d+1 = 3
vc_dim_linear_2d = empirical_vc_dimension_2d(max_points=5, n_random_configs=200)
print(f"Empirical VC dimension of linear classifiers in R^2: {vc_dim_linear_2d}")
print(f"Theoretical prediction: d+1 = 2+1 = 3")
assert vc_dim_linear_2d == 3, f"Expected 3, got {vc_dim_linear_2d}"
print("VC dimension estimate matches theory. ✓")

In [ ]:
# ── Visualise: 3 points shattered by a linear classifier ──────────────────────
# Show all 8 = 2^3 labellings of 3 points in R^2, each separable by a line
three_pts = np.array([[-1.0, 0.0], [1.0, 0.0], [0.0, 1.5]])

non_trivial_labellings = [
    lbls for lbls in itertools.product([0, 1], repeat=3)
    if 0 < sum(lbls) < 3
]
all_labellings = list(itertools.product([0, 1], repeat=3))  # all 8

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()

xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-1.5, 3, 200))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

for ax, labelling in zip(axes, all_labellings):
    labels_arr = np.array(labelling)
    ax.scatter(three_pts[:, 0], three_pts[:, 1],
               c=["steelblue" if l == 0 else "coral" for l in labels_arr],
               s=180, zorder=5, edgecolors="black", linewidths=1.5)
    for j, (pt, lbl) in enumerate(zip(three_pts, labels_arr)):
        ax.annotate(str(lbl), pt, textcoords="offset points",
                    xytext=(8, 5), fontsize=11, fontweight="bold")

    # Draw decision boundary if not all-same-class
    if 0 < labels_arr.sum() < 3:
        svm_v = SVC(kernel="linear", C=1e6)
        svm_v.fit(three_pts, labels_arr)
        Z = svm_v.predict(grid_pts).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.15,
                    cmap=ListedColormap(["steelblue", "coral"]))
        ax.contour(xx, yy, Z, levels=[0.5], colors="black", linewidths=1.5)

    ax.set_xlim(-3, 3)
    ax.set_ylim(-1.5, 3)
    ax.set_title(str(labelling), fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

patch0 = mpatches.Patch(color="steelblue", label="Class 0")
patch1 = mpatches.Patch(color="coral",     label="Class 1")
fig.legend(handles=[patch0, patch1], loc="lower center", ncol=2,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle(
    "All 8 Labellings of 3 Points Shattered by Linear Classifiers in R²\n"
    "→ VCdim(linear, R²) ≥ 3",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.show()
print("All 8 labellings are linearly separable. VCdim ≥ 3 confirmed visually.")

### 1.4 PAC Learning & Sample Complexity

A hypothesis class $\mathcal{H}$ is **PAC learnable** if there exists an algorithm $A$ and polynomial
$m(\varepsilon, \delta)$ such that: for any distribution $\mathcal{D}$ and any $\varepsilon, \delta > 0$,
given $m \geq m(\varepsilon, \delta)$ i.i.d. samples, $A$ outputs $h$ with probability $\geq 1-\delta$
satisfying $R(h) \leq \varepsilon$.

**Sample complexity (finite $\mathcal{H}$, consistent learner):** Setting the uniform convergence bound
equal to $\delta$ and solving for $n$:
$$n \geq \frac{1}{2\varepsilon^2}\left(\ln|\mathcal{H}| + \ln\frac{2}{\delta}\right)$$

**Sample complexity with VC dimension (infinite $\mathcal{H}$):** Via the Sauer–Shelah lemma, the
growth function $\Pi_{\mathcal{H}}(n) \leq (en/d)^d$ where $d = \text{VCdim}(\mathcal{H})$:
$$n \geq \frac{C}{\varepsilon^2}\left(d \ln\frac{1}{\varepsilon} + \ln\frac{1}{\delta}\right)$$

where $C \approx 4$ (the exact constant depends on the proof technique).

In [ ]:
# ── PAC Learning & Sample Complexity Bounds ───────────────────────────────────

def pac_sample_complexity_finite(
    epsilon: float,
    delta: float,
    hypothesis_class_size: int,
) -> int:
    '''Minimum samples for PAC learnability with a finite hypothesis class.

    Assumes a consistent learner (one that achieves zero training error).
    From: n >= (1 / 2*eps^2) * (ln|H| + ln(2/delta)).

    Args:
        epsilon: Acceptable excess error above Bayes error (accuracy tolerance).
        delta: Failure probability; bound holds with probability 1-delta.
        hypothesis_class_size: |H| — number of hypotheses in the class.

    Returns:
        Minimum integer sample count.
    '''
    log_term = math.log(hypothesis_class_size) + math.log(2.0 / delta)
    n_float  = log_term / (2.0 * epsilon ** 2)
    return math.ceil(n_float)


def pac_sample_complexity_vc(
    epsilon: float,
    delta: float,
    vc_dim: int,
    constant: float = 4.0,
) -> int:
    '''Minimum samples for PAC learnability parameterised by VC dimension.

    Uses the standard bound: n >= C/eps^2 * (d * ln(1/eps) + ln(1/delta))
    where d is the VC dimension and C ~ 4.

    Args:
        epsilon: Accuracy tolerance.
        delta: Confidence parameter.
        vc_dim: VC dimension of the hypothesis class.
        constant: Multiplicative constant C (theoretical default: 4.0).

    Returns:
        Minimum integer sample count.
    '''
    term = vc_dim * math.log(1.0 / epsilon) + math.log(1.0 / delta)
    n_float = (constant / epsilon ** 2) * term
    return math.ceil(n_float)


def vc_generalisation_bound(
    n: int,
    delta: float,
    vc_dim: int,
) -> float:
    '''Compute the VC-based generalisation bound complexity term.

    Via Sauer-Shelah: with prob >= 1-delta, R(h) <= R_hat(h) + bound_term.
    Uses the approximation: bound = sqrt((d*(ln(2n/d)+1) + ln(4/delta)) / n).

    Args:
        n: Number of training samples.
        delta: Confidence parameter.
        vc_dim: VC dimension.

    Returns:
        The sqrt(…) complexity penalty term.
    '''
    if n <= vc_dim:
        return 1.0  # bound is vacuous
    log_term = vc_dim * (math.log(2.0 * n / vc_dim) + 1.0) + math.log(4.0 / delta)
    return float(math.sqrt(log_term / n))


# ── Print sample complexity table ─────────────────────────────────────────────
print(f"PAC Sample Complexity (epsilon={EPSILON}, delta={DELTA}):")
print(f"\n{'Hypothesis class':30s}  {'VCdim':>6s}  {'n_required':>10s}")
print("-" * 52)

vc_models = [
    ("Linear (R^1)",            1 + 1),
    ("Linear (R^2)",            2 + 1),
    ("Linear (R^5)",            5 + 1),
    ("Linear (R^10)",          10 + 1),
    ("Linear (R^20)",          20 + 1),
    ("Decision Tree depth=3",   int(2**3 * 3)),
    ("Decision Tree depth=5",   int(2**5 * 5)),
    ("Decision Tree depth=10",  int(2**10 * 10)),
]

for model_name, vc_d in vc_models:
    n_req = pac_sample_complexity_vc(EPSILON, DELTA, vc_d)
    print(f"{model_name:30s}  {vc_d:6d}  {n_req:10,}")

print("\nKey insight: sample complexity grows roughly linearly with VC dimension.")

### 1.5 Growth Function and Sauer–Shelah Lemma

The **growth function** $\Pi_{\mathcal{H}}(n)$ counts the maximum number of distinct dichotomies
(labellings) that $\mathcal{H}$ can impose on any $n$ points:
$$\Pi_{\mathcal{H}}(n) = \max_{\{x_1,\ldots,x_n\} \subset \mathcal{X}} |\{(h(x_1),\ldots,h(x_n)) : h \in \mathcal{H}\}|$$

**Sauer–Shelah Lemma:** If $\text{VCdim}(\mathcal{H}) = d$, then for all $n \geq d$:
$$\Pi_{\mathcal{H}}(n) \leq \sum_{i=0}^{d} \binom{n}{i} \leq \left(\frac{en}{d}\right)^d$$

This polynomial bound (vs the exponential $2^n$ maximum) is what makes VC-dimension-based
generalisation bounds non-vacuous.

In [ ]:
# ── Growth Function & Sauer-Shelah Lemma ─────────────────────────────────────

def growth_function_exact(
    n: int,
    vc_dim: int,
) -> int:
    '''Compute the Sauer-Shelah upper bound on the growth function sum_binom.

    Evaluates: sum_{i=0}^{d} C(n, i) — the exact combinatorial bound from
    the Sauer-Shelah lemma.

    Args:
        n: Number of points.
        vc_dim: VC dimension of the hypothesis class.

    Returns:
        Integer upper bound on the growth function.
    '''
    return int(sum(math.comb(n, i) for i in range(min(vc_dim, n) + 1)))


def growth_function_poly(
    n: int,
    vc_dim: int,
) -> float:
    '''Compute the polynomial approximation (en/d)^d for the growth function.

    Args:
        n: Number of points.
        vc_dim: VC dimension.

    Returns:
        Polynomial upper bound on the growth function.
    '''
    if n < vc_dim:
        return float(2 ** n)  # trivial bound
    return float((math.e * n / vc_dim) ** vc_dim)


# ── Plot growth function: 2^n (max) vs Sauer-Shelah vs polynomial approx ──────
n_range = np.arange(1, 40)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for vc_d, color in [(3, "steelblue"), (5, "coral"), (10, "green")]:
    ss_vals  = [growth_function_exact(n, vc_d) for n in n_range]
    poly_vals = [growth_function_poly(n, vc_d) for n in n_range]

    axes[0].semilogy(n_range, ss_vals,   color=color, linestyle="-",
                     label=f"d={vc_d} Sauer-Shelah")
    axes[0].semilogy(n_range, poly_vals, color=color, linestyle="--",
                     alpha=0.6, label=f"d={vc_d} poly approx")

# 2^n maximum
axes[0].semilogy(n_range, [2**n for n in n_range], "k:",
                 linewidth=2, label="2^n (unconstrained)")
axes[0].set_xlabel("n (number of points)", fontsize=11)
axes[0].set_ylabel("Growth function Π_H(n) [log scale]", fontsize=11)
axes[0].set_title("Growth Function: Sauer-Shelah vs 2^n",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(True, alpha=0.3)

# VC-based generalisation bounds vs n for different VC dims
n_dense = np.arange(10, 2000, 20)
for vc_d, color in [(3, "steelblue"), (5, "coral"), (10, "green"), (20, "purple")]:
    bounds = [vc_generalisation_bound(n, DELTA, vc_d) for n in n_dense]
    axes[1].plot(n_dense, bounds, color=color, linewidth=2, label=f"d={vc_d}")

axes[1].axhline(EPSILON, color="black", linestyle="--", label=f"ε={EPSILON}")
axes[1].set_xlabel("Training samples n", fontsize=11)
axes[1].set_ylabel("VC Generalisation Bound", fontsize=11)
axes[1].set_title("How Many Samples to Get Bound ≤ ε?",
                  fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 0.8)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Sauer-Shelah Lemma & VC Generalisation Bounds",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Print the n where each bound crosses epsilon
print(f"Samples needed for VC bound <= {EPSILON} (delta={DELTA}):")
for vc_d in [3, 5, 10, 20]:
    n_req = pac_sample_complexity_vc(EPSILON, DELTA, vc_d)
    print(f"  VCdim={vc_d:3d}: n >= {n_req:,}")

---
## Part 2 — Empirical Validation Experiment

**Experiment design:** We sweep training set sizes $n \in \{20, 50, 100, \ldots, 3200\}$ and three
classifiers of increasing VC dimension.  For each $(n, \text{complexity})$ cell we repeat the
train-test split $N_{\text{rep}}=30$ times and measure the mean generalisation gap
$|\hat{R}(h) - R(h)|$.

**Theoretical prediction:** The VC bound predicts the gap should decay as $O(\sqrt{d/n})$, so
the gap should decrease with $n$ and increase with complexity.

**We then overlay the theoretical VC bound** on top of the empirical measurements to see how
tight (or loose) the theoretical guarantee is in practice.

In [ ]:
# ── Experiment: generalisation gap vs n for three classifiers ─────────────────

# We use a fixed large test set (5000 samples) to estimate true risk R(h)
X_full_exp, y_full_exp = make_classification(
    n_samples=10000,
    n_features=N_FEATURES,
    n_informative=N_FEATURES,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=SEED,
)

# Hold out 5000 samples as a fixed proxy for R(h)
X_pool, X_big_test, y_pool, y_big_test = train_test_split(
    X_full_exp, y_full_exp, test_size=0.5, random_state=SEED
)

# Classifiers with known approximate VC dimensions
# (In 2-D: linear=3, depth-3 tree ≈ moderate, depth-None tree ≈ large)
classifiers_vc = [
    ("Linear (d≈3)",         3,  LogisticRegression(max_iter=500, random_state=SEED)),
    ("Tree depth=3 (d≈24)", 24,  DecisionTreeClassifier(max_depth=3, random_state=SEED)),
    ("Tree depth=8 (d≈2048)", 2048, DecisionTreeClassifier(max_depth=8, random_state=SEED)),
]

gap_results: dict[str, dict] = {name: {"gaps": [], "vc": vc_d} for name, vc_d, _ in classifiers_vc}

for n_train in SAMPLE_SIZES:
    if n_train >= len(X_pool):
        continue
    for clf_name, vc_d, clf_proto in classifiers_vc:
        repeat_gaps: list[float] = []
        for rep in range(N_REPEATS):
            rng_rep = np.random.default_rng(SEED + rep * 1000 + n_train)
            idx = rng_rep.choice(len(X_pool), size=n_train, replace=False)
            X_tr_r, y_tr_r = X_pool[idx], y_pool[idx]

            clf = type(clf_proto)(**clf_proto.get_params())
            clf.fit(X_tr_r, y_tr_r)
            tr_err = empirical_risk(clf, X_tr_r, y_tr_r)
            te_err = empirical_risk(clf, X_big_test, y_big_test)
            repeat_gaps.append(abs(tr_err - te_err))

        gap_results[clf_name]["gaps"].append({
            "n": n_train,
            "mean_gap": float(np.mean(repeat_gaps)),
            "std_gap":  float(np.std(repeat_gaps)),
            "vc_bound": vc_generalisation_bound(n_train, DELTA, vc_d),
        })

print(f"Experiment complete. {len(SAMPLE_SIZES)} sample sizes × {len(classifiers_vc)} classifiers × {N_REPEATS} repeats.")
print(f"\nSample  | {classifiers_vc[0][0]:22s} | {classifiers_vc[1][0]:22s} | {classifiers_vc[2][0]:22s}")
print("-" * 90)
for i, n_train in enumerate(SAMPLE_SIZES):
    if n_train >= len(X_pool):
        continue
    row_parts = [f"{n_train:7d} "]
    for clf_name, _, _ in classifiers_vc:
        entry = gap_results[clf_name]["gaps"][i]
        row_parts.append(f"gap={entry['mean_gap']:.3f}±{entry['std_gap']:.3f}")
    print(" | ".join(row_parts))

In [ ]:
# ── Visualise: empirical gap vs n, with VC bound overlaid ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["steelblue", "coral", "green"]

for (clf_name, vc_d, _), color in zip(classifiers_vc, colors):
    gap_data = gap_results[clf_name]["gaps"]
    ns         = [d["n"]         for d in gap_data]
    mean_gaps  = [d["mean_gap"]  for d in gap_data]
    std_gaps   = [d["std_gap"]   for d in gap_data]
    vc_bounds  = [d["vc_bound"]  for d in gap_data]

    axes[0].errorbar(ns, mean_gaps, yerr=std_gaps,
                     fmt="-o", color=color, capsize=4,
                     label=f"{clf_name} (empirical)", linewidth=2)
    axes[0].plot(ns, vc_bounds, linestyle="--", color=color, alpha=0.5,
                 label=f"{clf_name} (VC bound)")

axes[0].set_xlabel("Training samples n", fontsize=11)
axes[0].set_ylabel("Generalisation gap |R_hat - R|", fontsize=11)
axes[0].set_title("Empirical Gap vs VC Bound", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=7, loc="upper right")
axes[0].grid(True, alpha=0.3)

# Log-log plot: gap ~ C * n^{-0.5}
for (clf_name, vc_d, _), color in zip(classifiers_vc, colors):
    gap_data = gap_results[clf_name]["gaps"]
    ns        = np.array([d["n"]        for d in gap_data])
    mean_gaps = np.array([d["mean_gap"] for d in gap_data])
    axes[1].loglog(ns, mean_gaps, "-o", color=color, label=clf_name, linewidth=2)

# Overlay O(1/sqrt(n)) reference line
ns_ref = np.array(SAMPLE_SIZES, dtype=float)
axes[1].loglog(ns_ref, 1.2 / np.sqrt(ns_ref), "k--",
               linewidth=1.5, label="O(1/√n) reference")
axes[1].set_xlabel("Training samples n (log)", fontsize=11)
axes[1].set_ylabel("|R_hat - R| (log)", fontsize=11)
axes[1].set_title("Log-Log: Gap Decay Rate", fontsize=12, fontweight="bold")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, which="both")

plt.suptitle("Empirical Generalisation Gap vs Theoretical VC Bounds",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## Part 3 — Application to Real Classification Problems

We now apply the theoretical framework to a practical setting: **given a fixed budget of $n$ samples,
which model complexity should we choose?**  The PAC bound predicts that:
- High-capacity models (large VC dim) require many samples to close the gap
- Low-capacity models may underfit but have tight generalisation guarantees

We train all three classifiers at each sample size and measure **actual test accuracy** to see
whether the theory-driven complexity recommendation aligns with empirical performance.

In [ ]:
# ── Practical experiment: which model wins at each data budget? ───────────────

# Use a 20-feature dataset (harder, more realistic)
X_real, y_real = make_classification(
    n_samples=8000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=2,
    random_state=SEED,
)
X_real_pool, X_real_test, y_real_pool, y_real_test = train_test_split(
    X_real, y_real, test_size=0.4, random_state=SEED
)

models_real = [
    ("Logistic Regression", LogisticRegression(max_iter=500, random_state=SEED)),
    ("Tree depth=3", DecisionTreeClassifier(max_depth=3, random_state=SEED)),
    ("Tree depth=5", DecisionTreeClassifier(max_depth=5, random_state=SEED)),
    ("Tree (no limit)", DecisionTreeClassifier(max_depth=None, random_state=SEED)),
]

sample_sizes_real = [50, 100, 200, 400, 800, 1600, 3200]
acc_results: dict[str, list[float]] = {name: [] for name, _ in models_real}

for n_tr in sample_sizes_real:
    rng_r = np.random.default_rng(SEED)
    idx = rng_r.choice(len(X_real_pool), size=n_tr, replace=False)
    X_tr_r, y_tr_r = X_real_pool[idx], y_real_pool[idx]

    for name, clf_template in models_real:
        clf = type(clf_template)(**clf_template.get_params())
        clf.fit(X_tr_r, y_tr_r)
        acc = accuracy_score(y_real_test, clf.predict(X_real_test))
        acc_results[name].append(acc)

# Display results
print(f"Test accuracy vs training set size (20-feature classification):")
print(f"{'n':>6s}  ", end="")
for name, _ in models_real:
    print(f"{name:22s}", end="  ")
print()
print("-" * 100)
for i, n_tr in enumerate(sample_sizes_real):
    print(f"{n_tr:6d}  ", end="")
    for name, _ in models_real:
        print(f"{acc_results[name][i]:22.3f}", end="  ")
    print()

In [ ]:
# ── Plot: test accuracy vs n for each model ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
model_colors = ["steelblue", "coral", "green", "purple"]

for (name, _), color in zip(models_real, model_colors):
    axes[0].plot(sample_sizes_real, acc_results[name],
                 marker="o", label=name, color=color, linewidth=2)

axes[0].set_xlabel("Training samples", fontsize=11)
axes[0].set_ylabel("Test Accuracy", fontsize=11)
axes[0].set_title("Test Accuracy vs Data Budget", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Cross-over plot: which model is best at each budget?
best_model_per_budget = [
    max(range(len(models_real)),
        key=lambda k: acc_results[models_real[k][0]][i])
    for i in range(len(sample_sizes_real))
]
best_model_names = [models_real[k][0] for k in best_model_per_budget]
best_accs = [
    acc_results[models_real[k][0]][i]
    for i, k in enumerate(best_model_per_budget)
]

axes[1].bar(range(len(sample_sizes_real)), best_accs,
            color=[model_colors[k] for k in best_model_per_budget],
            edgecolor="white", width=0.6)
axes[1].set_xticks(range(len(sample_sizes_real)))
axes[1].set_xticklabels([str(n) for n in sample_sizes_real], rotation=30)
axes[1].set_xlabel("Training samples", fontsize=11)
axes[1].set_ylabel("Best test accuracy", fontsize=11)
axes[1].set_title("Best Model at Each Data Budget",
                  fontsize=12, fontweight="bold")
axes[1].set_ylim(0.5, 1.0)
for i, (acc, name) in enumerate(zip(best_accs, best_model_names)):
    axes[1].text(i, acc + 0.005, name[:12], ha="center", va="bottom",
                 fontsize=7, rotation=45)
axes[1].grid(True, axis="y", alpha=0.3)

# Legend patches
handles = [mpatches.Patch(color=model_colors[k], label=models_real[k][0])
           for k in range(len(models_real))]
axes[1].legend(handles=handles, fontsize=8, loc="lower right")

plt.suptitle("Model Selection Under Data Budget Constraints",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("Observation: low-capacity models win at small n; high-capacity win at large n.")

---
## Part 4 — Theory vs Practice: Where Bounds Are Tight (and Where They Aren't)

A well-known frustration with VC-dimension-based bounds is that they are often **very loose** in
practice.  The bounds are worst-case guarantees that hold for *all* distributions; real datasets
have benign structure that the theory does not exploit.

In this section we:
1. Quantify the **tightness ratio** (empirical gap / VC bound) across all our experiments.
2. Show how the Hoeffding bound quality degrades as $\varepsilon$ shrinks.
3. Discuss the **double descent phenomenon** — a modern result that challenges classical VC intuition.

In [ ]:
# ── Bound tightness: empirical gap / VC theoretical bound ─────────────────────
print("Bound Tightness Ratio = empirical_gap / VC_bound (closer to 1.0 = tighter bound)")
print(f"\n{'Classifier':28s}  {'n':>6s}  {'Emp. gap':>9s}  {'VC bound':>9s}  {'Tightness':>10s}")
print("-" * 70)

tightness_rows: list = []
for clf_name, vc_d, _ in classifiers_vc:
    for entry in gap_results[clf_name]["gaps"]:
        ratio = (
            entry["mean_gap"] / entry["vc_bound"]
            if entry["vc_bound"] > 0 else float("nan")
        )
        tightness_rows.append((clf_name, entry["n"], entry["mean_gap"],
                                entry["vc_bound"], ratio))
        print(
            f"{clf_name:28s}  {entry['n']:6d}  "
            f"{entry['mean_gap']:9.4f}  "
            f"{entry['vc_bound']:9.4f}  "
            f"{ratio:10.4f}"
        )

tightness_df = pd.DataFrame(
    tightness_rows,
    columns=["Classifier", "n", "Emp. gap", "VC bound", "Tightness"]
)
mean_tight = tightness_df.groupby("Classifier")["Tightness"].mean()
print(f"\nMean tightness by classifier:")
for clf_name, tight in mean_tight.items():
    print(f"  {clf_name:28s}: {tight:.4f}")
print("\nConclusion: VC bounds are loose by 1-2 orders of magnitude on benign data.")

In [ ]:
# ── Hoeffding bound quality vs epsilon ────────────────────────────────────────
# For what epsilon does the Hoeffding bound become practically useful?
# Useful = bound probability < 0.1 (90% confidence that deviation < epsilon)

epsilon_range = np.linspace(0.01, 0.5, 200)
n_values = [50, 100, 200, 500, 1000, 5000]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pal = plt.cm.viridis(np.linspace(0, 0.85, len(n_values)))

for n_val, color in zip(n_values, pal):
    bounds = [hoeffding_bound(n_val, eps) for eps in epsilon_range]
    axes[0].semilogy(epsilon_range, bounds, color=color, linewidth=2, label=f"n={n_val}")

axes[0].axhline(0.05, color="black", linestyle="--", label="δ=0.05")
axes[0].axhline(0.10, color="gray",  linestyle=":",  label="δ=0.10")
axes[0].set_xlabel("Epsilon (tolerance)", fontsize=11)
axes[0].set_ylabel("Hoeffding bound P(|mean-E|≥ε) [log]", fontsize=11)
axes[0].set_title("Hoeffding Bound vs ε for Varying n",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8, loc="upper right")
axes[0].grid(True, alpha=0.3)

# Sample complexity comparison: finite H vs VC dimension bound
vc_dims_compare = [3, 5, 10, 20, 50, 100]
epsilons_compare = np.linspace(0.02, 0.30, 100)
delta_compare = 0.05

for vc_d_c, color in zip([3, 10, 50], ["steelblue", "coral", "green"]):
    n_reqs = [pac_sample_complexity_vc(eps, delta_compare, vc_d_c)
              for eps in epsilons_compare]
    axes[1].semilogy(epsilons_compare, n_reqs, color=color,
                     linewidth=2, label=f"VCdim={vc_d_c}")

axes[1].set_xlabel("Epsilon (accuracy tolerance)", fontsize=11)
axes[1].set_ylabel("Required samples n (log scale)", fontsize=11)
axes[1].set_title("PAC Sample Complexity vs ε",
                  fontsize=11, fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Hoeffding Bound Quality & PAC Sample Complexity",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Double Descent: Classical VC theory predicts U-shape, modern models show more ─
# Classical VC theory: as capacity increases, test error first decreases (underfit→fit),
# then increases (overfit). Modern over-parameterised models show a SECOND descent.
# We demonstrate the classical U-shape here; true double descent requires
# interpolation-regime models (neural networks, covered in Module 5+).

X_dd, y_dd = make_classification(
    n_samples=300, n_features=N_FEATURES, n_informative=N_FEATURES,
    n_redundant=0, n_clusters_per_class=1, random_state=SEED
)
X_dd_tr, X_dd_te, y_dd_tr, y_dd_te = train_test_split(
    X_dd, y_dd, test_size=0.5, random_state=SEED
)

# Vary tree depth = proxy for capacity / VC dimension
depths = list(range(1, 21))
train_accs_dd: list[float] = []
test_accs_dd: list[float]  = []

for depth in depths:
    clf_dd = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    clf_dd.fit(X_dd_tr, y_dd_tr)
    train_accs_dd.append(accuracy_score(y_dd_tr, clf_dd.predict(X_dd_tr)))
    test_accs_dd.append(accuracy_score(y_dd_te, clf_dd.predict(X_dd_te)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_accs_dd, "o-", color="steelblue", label="Train accuracy", linewidth=2)
ax.plot(depths, test_accs_dd,  "s-", color="coral",     label="Test accuracy",  linewidth=2)
ax.set_xlabel("Tree Depth (proxy for VC dimension)", fontsize=11)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_title(
    "Classical Bias-Variance U-Shape: Test Error vs Model Capacity\n"
    "(Double descent is the extension beyond the interpolation threshold — see 4-07)",
    fontsize=11, fontweight="bold",
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Mark optimal depth
best_depth = depths[int(np.argmax(test_accs_dd))]
best_acc = max(test_accs_dd)
ax.axvline(best_depth, color="green", linestyle="--",
           label=f"Best depth={best_depth} (acc={best_acc:.3f})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Optimal test accuracy at depth={best_depth}: {best_acc:.3f}")
print("Train accuracy reaches 1.0 at large depths (perfect memorisation).")
print("VC theory correctly predicts test error rises after the optimal capacity.")

### Rademacher Complexity: A Tighter Alternative to VC Dimension

Rademacher complexity directly measures how well a hypothesis class can fit **random labels** — a more data-dependent and often tighter measure than VC dimension.  The generalisation bound using Rademacher complexity is:

$$R(h) \leq \hat{R}(h) + 2\hat{\mathcal{R}}_n(\mathcal{H}) + \sqrt{\frac{\ln(2/\delta)}{2n}}$$

where $\hat{\mathcal{R}}_n(\mathcal{H})$ is the empirical Rademacher complexity estimated on the training sample.


In [ ]:
# ── Rademacher Complexity ─────────────────────────────────────────────────────
# Rademacher complexity is a tighter alternative to VC dimension.  Instead of
# asking how many points can be shattered, it measures how well a hypothesis
# class correlates with *random noise* on the training set.
#
# Empirical Rademacher complexity of H on sample S = {x_1, ..., x_n}:
#
#   R_n(H) = E_sigma [ sup_{h in H} (1/n) sum_i sigma_i * h(x_i) ]
#
# where sigma_i ~ Uniform({-1, +1}) are Rademacher random variables.
#
# Generalisation bound (Rademacher): with probability >= 1-delta, for all h in H:
#   R(h) <= R_hat(h) + 2 * R_n(H) + sqrt(ln(2/delta) / (2n))


def empirical_rademacher_complexity(
    X: np.ndarray,
    predict_fn: Callable,
    n_rademacher: int = 200,
    seed: int = SEED,
) -> float:
    '''Estimate empirical Rademacher complexity for a classifier on sample X.

    Draws n_rademacher random sign vectors sigma and measures the average
    maximum correlation between sigma and the model's predictions.  The
    supremum over H is approximated by using the actual fitted classifier.

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        predict_fn: Callable that takes X and returns predictions in {-1, +1}.
        n_rademacher: Number of Monte Carlo Rademacher sign vectors.
        seed: Random seed.

    Returns:
        Scalar estimate of empirical Rademacher complexity.
    '''
    rng_r = np.random.default_rng(seed)
    n = X.shape[0]
    y_hat = predict_fn(X).astype(float)  # (n,) in {-1, +1}

    correlations: list[float] = []
    for _ in range(n_rademacher):
        sigma = rng_r.choice([-1.0, 1.0], size=n)
        corr  = float(np.mean(sigma * y_hat))
        correlations.append(abs(corr))

    return float(np.mean(correlations))


def rademacher_generalisation_bound(
    n: int,
    delta: float,
    rademacher_complexity: float,
) -> float:
    '''Compute the Rademacher-based generalisation bound complexity term.

    With prob >= 1-delta: R(h) <= R_hat(h) + 2*R_n(H) + sqrt(ln(2/delta)/(2n)).

    Args:
        n: Number of training samples.
        delta: Confidence parameter.
        rademacher_complexity: Estimated empirical Rademacher complexity R_n(H).

    Returns:
        Total complexity penalty (2*R_n + sqrt term).
    '''
    sqrt_term = math.sqrt(math.log(2.0 / delta) / (2.0 * n))
    return float(2.0 * rademacher_complexity + sqrt_term)


# ── Compare Rademacher vs VC bound for three classifiers ──────────────────────
print("Rademacher vs VC Generalisation Bounds (n=200 training samples, delta=0.05):")
print(f"\n{'Classifier':28s}  {'Rad. Cmplx':>10s}  {'Rad. Bound':>10s}  {'VC Bound':>10s}  {'Emp. Gap':>10s}")
print("-" * 75)

n_compare = 200
for clf_name, vc_d, clf_template in classifiers_vc:
    rng_c = np.random.default_rng(SEED)
    idx = rng_c.choice(len(X_pool), size=n_compare, replace=False)
    X_c, y_c = X_pool[idx], y_pool[idx]

    clf_c = type(clf_template)(**clf_template.get_params())
    clf_c.fit(X_c, y_c)

    # Convert {0,1} predictions to {-1,+1} for Rademacher
    # We capture clf_c via a default-argument lambda — no nested def needed
    predict_pm1 = lambda Xq, _clf=clf_c: 2.0 * _clf.predict(Xq).astype(float) - 1.0

    rad_c = empirical_rademacher_complexity(
        X_c, predict_pm1, n_rademacher=200, seed=SEED
    )
    rad_bound_c = rademacher_generalisation_bound(n_compare, DELTA, rad_c)
    vc_bound_c  = vc_generalisation_bound(n_compare, DELTA, vc_d)

    # Empirical gap at n=200 (from earlier experiment)
    gap_entry = next(
        (e for e in gap_results[clf_name]["gaps"] if e["n"] == n_compare), None
    )
    emp_gap = gap_entry["mean_gap"] if gap_entry else float("nan")

    print(
        f"{clf_name:28s}  {rad_c:10.4f}  "
        f"{rad_bound_c:10.4f}  {vc_bound_c:10.4f}  {emp_gap:10.4f}"
    )

print("\nRademacher bounds are tighter than VC bounds for low-complexity classifiers.")


### Empirical Verification of the PAC Bound

The PAC bound makes a testable prediction: if $n \geq m(\varepsilon, \delta)$, then the fraction of experiments where the gap exceeds $\varepsilon$ should be at most $\delta$.  We verify this by running 300 Monte Carlo trials at each training size.


In [ ]:
# ── PAC Bound Empirical Verification ─────────────────────────────────────────
# The PAC bound says: if n >= pac_sample_complexity(eps, delta, |H|), then
# with probability >= 1-delta, ALL h in H satisfy |R_hat - R| <= eps.
# We verify this by running many train/test experiments and counting violations.


def pac_bound_violation_rate(
    n_train: int,
    epsilon: float,
    n_trials: int = 500,
    seed: int = SEED,
) -> dict[str, float]:
    '''Estimate empirical violation rate of the PAC generalisation bound.

    For each trial: (1) draw n_train samples, (2) fit LogisticRegression,
    (3) measure generalisation gap, (4) check if gap > epsilon.
    Returns the fraction of trials where the bound is violated.

    Args:
        n_train: Training set size per trial.
        epsilon: Deviation tolerance for the PAC bound.
        n_trials: Number of Monte Carlo trials.
        seed: Random seed.

    Returns:
        Dict with keys ``"violation_rate"``, ``"mean_gap"``, ``"std_gap"``,
        ``"pac_required_n"``.
    '''
    rng_pac = np.random.default_rng(seed)
    gaps: list[float] = []

    for trial in range(n_trials):
        idx = rng_pac.choice(len(X_pool), size=n_train, replace=False)
        X_tr_t, y_tr_t = X_pool[idx], y_pool[idx]
        clf_t = LogisticRegression(max_iter=300, random_state=int(seed) + trial)
        clf_t.fit(X_tr_t, y_tr_t)
        tr_err = empirical_risk(clf_t, X_tr_t, y_tr_t)
        te_err = empirical_risk(clf_t, X_big_test, y_big_test)
        gaps.append(abs(tr_err - te_err))

    violation_rate = float(np.mean(np.array(gaps) > epsilon))
    return {
        "violation_rate": violation_rate,
        "mean_gap":       float(np.mean(gaps)),
        "std_gap":        float(np.std(gaps)),
        "pac_required_n": pac_sample_complexity_vc(epsilon, DELTA, vc_dim=3),
    }


# Sweep n_train and check empirical violation rate vs PAC-predicted delta
eps_verify_pac = 0.10
print(f"PAC Bound Verification (epsilon={eps_verify_pac}, delta={DELTA}):")
print(f"PAC theory requires n >= {pac_sample_complexity_vc(eps_verify_pac, DELTA, 3):,} samples")
print(f"\n{'n_train':>8s}  {'Viol. rate':>11s}  {'PAC delta':>10s}  {'Mean gap':>9s}  {'Bound tight?'}")
print("-" * 62)

for n_tr_v in [30, 60, 100, 200, 400, 800]:
    pac_res = pac_bound_violation_rate(n_tr_v, eps_verify_pac, n_trials=300, seed=SEED)
    pac_delta_pred = uniform_convergence_bound(n_tr_v, eps_verify_pac,
                                               hypothesis_class_size=10**6)
    tight = "✓ (viol <= delta)" if pac_res["violation_rate"] <= DELTA else "x (viol > delta)"
    print(
        f"{n_tr_v:8d}  "
        f"{pac_res['violation_rate']:11.4f}  "
        f"{min(pac_delta_pred, 1.0):10.4f}  "
        f"{pac_res['mean_gap']:9.4f}  "
        f"{tight}"
    )

print("\nNote: High violation rates at small n because the data has structure")
print("(clusters, correlations) not captured by worst-case PAC bounds.")


### Structural Risk Minimisation (SRM)

SRM (Vapnik, 1998) is the theoretical basis for regularisation and model selection.  Instead of choosing the model with the lowest training error, SRM selects the model that minimises the **total generalisation bound**:

$$k^* = \underset{k}{\arg\min} \left[ \min_{h \in \mathcal{H}_k} \hat{R}(h) + \text{complexity\_bound}(n, \delta, d_k) \right]$$

This provides a principled justification for preferring simpler models when data is scarce.


In [ ]:
# ── Structural Risk Minimisation (SRM) ───────────────────────────────────────
# SRM (Vapnik, 1998) formalises model selection using generalisation bounds:
# instead of just minimising training error, choose the hypothesis class H_k
# that minimises R_hat(h) + complexity_penalty(k, n, delta).
#
# This is the theoretical justification for regularisation and cross-validation.
#
# For a nested sequence H_1 ⊆ H_2 ⊆ ... ⊆ H_k with increasing VC dimensions,
# SRM selects k* = argmin_k [ min_{h in H_k} R_hat(h) + bound(n, delta, d_k) ]


def srm_model_selection(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    candidate_models: list,
    delta: float = DELTA,
) -> tuple[int, float]:
    '''Select the model that minimises the SRM objective (train error + bound).

    Each candidate is a (model_name, vc_dim, fitted_classifier) tuple.
    The SRM objective is: train_error + vc_generalisation_bound(n, delta, vc_dim).

    Args:
        X_tr: Training feature matrix.
        y_tr: Training labels.
        X_val: Validation feature matrix.
        y_val: Validation labels (used only for reporting, not selection).
        candidate_models: List of (name, vc_dim, classifier) tuples.
        delta: Confidence parameter for the bound.

    Returns:
        Tuple of (best_model_index, best_srm_objective_value).
    '''
    n = len(y_tr)
    srm_objectives: list[float] = []
    for name, vc_d, clf in candidate_models:
        tr_err = empirical_risk(clf, X_tr, y_tr)
        bound  = vc_generalisation_bound(n, delta, vc_d)
        srm_objectives.append(tr_err + bound)
    best_idx = int(np.argmin(srm_objectives))
    return best_idx, float(srm_objectives[best_idx])


# ── Compare SRM selection vs CV selection vs naive (lowest train error) ────────
X_srm, y_srm = make_classification(
    n_samples=2000, n_features=N_FEATURES, n_informative=N_FEATURES,
    n_redundant=0, n_clusters_per_class=1, random_state=SEED + 42
)
X_srm_tr, X_srm_test, y_srm_tr, y_srm_test = train_test_split(
    X_srm, y_srm, test_size=0.3, random_state=SEED
)

srm_candidates = [
    ("LR (d=3)",           3,  LogisticRegression(max_iter=500, random_state=SEED)),
    ("Tree d=2 (~d=4)",    4,  DecisionTreeClassifier(max_depth=2, random_state=SEED)),
    ("Tree d=4 (~d=16)",  16,  DecisionTreeClassifier(max_depth=4, random_state=SEED)),
    ("Tree d=6 (~d=64)",  64,  DecisionTreeClassifier(max_depth=6, random_state=SEED)),
    ("Tree d=None(large)", 512, DecisionTreeClassifier(max_depth=None, random_state=SEED)),
]

for i, (name, vc_d, clf) in enumerate(srm_candidates):
    clf.fit(X_srm_tr, y_srm_tr)

best_srm_idx, best_srm_obj = srm_model_selection(
    X_srm_tr, y_srm_tr, X_srm_test, y_srm_test, srm_candidates
)
best_naive_idx = int(np.argmin(
    [empirical_risk(clf, X_srm_tr, y_srm_tr) for _, _, clf in srm_candidates]
))

print("SRM vs Naive Model Selection:")
print(f"\n{'Model':22s}  {'Train err':>9s}  {'VC bound':>9s}  {'SRM obj':>9s}  {'Test acc':>9s}")
print("-" * 68)
for i, (name, vc_d, clf) in enumerate(srm_candidates):
    tr_err = empirical_risk(clf, X_srm_tr, y_srm_tr)
    bound  = vc_generalisation_bound(len(y_srm_tr), DELTA, vc_d)
    srm_obj = tr_err + bound
    te_acc  = accuracy_score(y_srm_test, clf.predict(X_srm_test))
    marker  = " ← SRM" if i == best_srm_idx else (" ← Naive" if i == best_naive_idx else "")
    print(f"{name:22s}  {tr_err:9.4f}  {bound:9.4f}  {srm_obj:9.4f}  {te_acc:9.4f}{marker}")

best_srm_acc  = accuracy_score(y_srm_test, srm_candidates[best_srm_idx][2].predict(X_srm_test))
best_naive_acc = accuracy_score(y_srm_test, srm_candidates[best_naive_idx][2].predict(X_srm_test))
print(f"\nSRM selection test accuracy   : {best_srm_acc:.4f}")
print(f"Naive (min train err) test acc: {best_naive_acc:.4f}")
print("SRM protects against selecting an overfit model by penalising high VC dimension.")


### Finite Hypothesis Class: When the Bound Is Exact

For a *finite* hypothesis class $|\mathcal{H}| = K$, the uniform convergence bound
is directly usable (no Sauer–Shelah approximation needed).  We construct a concrete
example with axis-aligned threshold classifiers and measure bound tightness as $K$ grows.
Then we visualise the generalisation bound and PAC sample complexity as heatmaps over
the $(n, d)$ and $(\varepsilon, d)$ grids.


In [ ]:
# ── Finite Hypothesis Class: Theory Is Exact When |H| Is Small ───────────────
# For a finite hypothesis class H = {h_1, ..., h_K}, the uniform convergence
# bound is: P(sup_h |R_hat - R| >= eps) <= 2*K*exp(-2*n*eps^2).
# This bound is tighter than the VC bound when K is small and known.
#
# We construct a concrete finite class of axis-aligned threshold classifiers
# on 1-D data:  h_theta(x) = 1[x >= theta],  theta in {t_1, ..., t_K}.


def build_threshold_classifiers(
    thresholds: np.ndarray,
) -> list:
    '''Build a list of threshold classifier functions for 1-D data.

    Each classifier h_theta predicts 1 if x >= theta, else 0.

    Args:
        thresholds: Array of threshold values.

    Returns:
        List of Callables (numpy_array -> numpy_array).
    '''
    # Use list comprehension with default-arg lambdas to avoid closure capture issues
    classifiers: list = [
        (lambda t: (lambda X_1d: (X_1d >= t).astype(int)))(float(theta))
        for theta in thresholds
    ]
    return classifiers


def erm_finite_class(
    X_train_1d: np.ndarray,
    y_train: np.ndarray,
    classifiers: list,
) -> tuple[int, float]:
    '''Select the ERM hypothesis from a finite class on 1-D training data.

    Args:
        X_train_1d: 1-D feature array of shape (n_samples,).
        y_train: Binary labels of shape (n_samples,).
        classifiers: List of Callable threshold classifiers.

    Returns:
        Tuple of (best_index, best_train_error).
    '''
    best_idx   = 0
    best_error = float("inf")
    for i, clf in enumerate(classifiers):
        err = float(np.mean(clf(X_train_1d) != y_train))
        if err < best_error:
            best_error = err
            best_idx   = i
    return best_idx, best_error


# Generate 1-D data: positive class has higher values
rng_1d = np.random.default_rng(SEED)
n_large = 2000
X_1d_full = rng_1d.uniform(-3, 3, size=n_large)
true_theta = 0.5   # ground truth threshold
y_1d_full  = (X_1d_full >= true_theta).astype(int)
y_1d_full ^= (rng_1d.random(n_large) < 0.05).astype(int)  # 5% label noise

# Threshold candidates: 40 evenly spaced values in [-2, 2]
K_values = [5, 10, 20, 40, 80]
test_idxs = rng_1d.choice(n_large, size=500, replace=False)
train_mask = np.ones(n_large, dtype=bool)
train_mask[test_idxs] = False
X_1d_test, y_1d_test = X_1d_full[test_idxs], y_1d_full[test_idxs]

print("Finite hypothesis class: threshold classifiers on 1-D data")
print(f"{'K (|H|)':>8s}  {'Train err':>9s}  {'Test err':>9s}  {'Gap':>7s}  "
      f"{'Hoeff bound':>12s}  {'Tightness':>10s}")
print("-" * 65)

finite_rows: list = []
for K in K_values:
    thresholds = np.linspace(-2.0, 2.0, K)
    h_class = build_threshold_classifiers(thresholds)

    pool_idx = np.where(train_mask)[0]
    n_finite_train = 300
    sel = rng_1d.choice(pool_idx, size=n_finite_train, replace=False)
    X_tr_f, y_tr_f = X_1d_full[sel], y_1d_full[sel]

    best_i, tr_err_f = erm_finite_class(X_tr_f, y_tr_f, h_class)
    best_clf = h_class[best_i]
    te_err_f = float(np.mean(best_clf(X_1d_test) != y_1d_test))
    gap_f    = abs(tr_err_f - te_err_f)
    uc_bound = uniform_convergence_bound(n_finite_train, 0.05, K)
    tightness = gap_f / uc_bound if uc_bound > 0 else float("nan")

    finite_rows.append((K, tr_err_f, te_err_f, gap_f, uc_bound, tightness))
    print(f"{K:8d}  {tr_err_f:9.4f}  {te_err_f:9.4f}  {gap_f:7.4f}  "
          f"{uc_bound:12.4f}  {tightness:10.4f}")

print("\nLarger K = richer class = looser bound — confirms theory.")

# ── Heatmap: VC generalisation bound across (n, d) grid ──────────────────────
n_grid = np.array([50, 100, 200, 500, 1000, 2000, 5000])
d_grid = np.array([1, 3, 5, 10, 20, 50, 100])

bound_matrix = np.zeros((len(d_grid), len(n_grid)))
for i, d_val in enumerate(d_grid):
    for j, n_val in enumerate(n_grid):
        bound_matrix[i, j] = vc_generalisation_bound(n_val, DELTA, d_val)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(
    bound_matrix, aspect="auto", cmap="coolwarm",
    vmin=0, vmax=min(1.0, bound_matrix.max()),
)
axes[0].set_xticks(range(len(n_grid)))
axes[0].set_xticklabels([str(n) for n in n_grid], rotation=30)
axes[0].set_yticks(range(len(d_grid)))
axes[0].set_yticklabels([str(d) for d in d_grid])
axes[0].set_xlabel("Training samples n", fontsize=11)
axes[0].set_ylabel("VC dimension d", fontsize=11)
axes[0].set_title("VC Generalisation Bound\n(red = loose, blue = tight)",
                   fontsize=11, fontweight="bold")
plt.colorbar(im, ax=axes[0], label="Bound value")

# Annotate cells with bound value
for i in range(len(d_grid)):
    for j in range(len(n_grid)):
        val = bound_matrix[i, j]
        color = "white" if val > 0.4 else "black"
        axes[0].text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=7, color=color)

# Sample complexity surface: n_required vs (eps, d)
eps_range_heat = [0.05, 0.10, 0.15, 0.20, 0.25]
d_range_heat   = [3, 5, 10, 20, 50]
n_req_matrix = np.array([
    [pac_sample_complexity_vc(eps, DELTA, d) for d in d_range_heat]
    for eps in eps_range_heat
])

im2 = axes[1].imshow(np.log10(n_req_matrix), aspect="auto", cmap="viridis")
axes[1].set_xticks(range(len(d_range_heat)))
axes[1].set_xticklabels([str(d) for d in d_range_heat])
axes[1].set_yticks(range(len(eps_range_heat)))
axes[1].set_yticklabels([str(e) for e in eps_range_heat])
axes[1].set_xlabel("VC dimension d", fontsize=11)
axes[1].set_ylabel("Epsilon (tolerance)", fontsize=11)
axes[1].set_title("PAC Sample Complexity log10(n_required)\n"
                   "smaller eps + larger d → exponentially more data",
                   fontsize=11, fontweight="bold")
plt.colorbar(im2, ax=axes[1], label="log10(n_required)")

for i in range(len(eps_range_heat)):
    for j in range(len(d_range_heat)):
        axes[1].text(j, i, f"{n_req_matrix[i,j]:,.0f}", ha="center", va="center",
                     fontsize=8, color="white")

plt.suptitle("Finite Hypothesis Class & Generalisation Bound Heatmaps",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
# Print summary of bound vs sample size relationship
print("VC Bound decay: as n doubles, bound decreases by ~1/sqrt(2) = 0.707x")
for n_val_s in [100, 200, 400, 800, 1600]:
    b = vc_generalisation_bound(n_val_s, DELTA, vc_dim=3)
    print(f"  n={n_val_s:5d}  VC bound={b:.4f}")
print("Finite class and bound heatmap analysis complete.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Empirical risk is not true risk.** The fundamental learning theory question is when
   $\hat{R}(h) \approx R(h)$ — Hoeffding's inequality quantifies this for a single hypothesis,
   and the union bound extends it to an entire hypothesis class $\mathcal{H}$.

2. **VC dimension is the right measure of hypothesis class complexity.** It replaces the intractable
   $|\mathcal{H}| = \infty$ with a finite combinatorial quantity — the maximum number of points that
   the class can shatter.  For linear classifiers in $\mathbb{R}^d$, VCdim $= d+1$, which is why
   more features require more data.

3. **Sample complexity scales linearly with VC dimension.** The PAC bound
   $n \geq \frac{C}{\varepsilon^2}(d \ln \frac{1}{\varepsilon} + \ln \frac{1}{\delta})$
   explains the empirical rule of thumb: "get at least 10× the VC dimension in training samples."

4. **VC bounds are loose in practice.** The theoretical bound is a worst-case guarantee over all
   distributions.  In experiments, empirical generalisation gaps are 10–100× smaller than the VC
   bound predicts.  The bound is a *qualitative* guide, not a tight *quantitative* prediction.

5. **Classical VC theory predicts a U-shaped test error curve** (under- to over-fitting) as capacity
   increases.  Modern over-parameterised models exhibit a second descent beyond the interpolation
   threshold — the *double descent* phenomenon — challenging classical VC intuitions.

### What's Next

→ **04-07 (Bias-Variance Decomposition & ML Debugging)** formalises the U-shaped error curve
through the bias-variance-noise decomposition and introduces the systematic CS229 debugging recipe.

### Going Further

- Shalev-Shwartz & Ben-David, *Understanding Machine Learning*, Chapters 2–6 (free PDF)
- Blumer et al. (1989), "Learnability and the Vapnik-Chervonenkis dimension"
- Belkin et al. (2019), "Reconciling modern machine learning and the bias-variance trade-off" (double descent)

In [ ]:
# ── Part 5: Comprehensive Theory Summary ─────────────────────────────────
# Collect key quantities from this notebook into a consolidated reference.

print("=" * 70)
print("LEARNING THEORY SUMMARY -- KEY QUANTITIES")
print("=" * 70)

# 1. Generalisation bounds
print("")
print("1. Generalisation bounds at n=500, delta=0.05")
print(f"   {'Bound type':<30s}  {'Value':>10s}")
print(f"   {'-'*44}")
n_ref, delta_ref, K_ref, d_ref = 500, 0.05, 20, 3
hb_val  = hoeffding_bound(n_ref, 0.10)
uc_val  = uniform_convergence_bound(n_ref, 0.10, K_ref)
gf_val  = generalisation_bound_finite(n_ref, delta_ref, K_ref)
vc_val  = vc_generalisation_bound(n_ref, delta_ref, d_ref)
print(f"   {'Hoeffding (eps=0.10)':<30s}  {hb_val:10.4f}")
print(f"   {'UniformConv (|H|=20, eps=0.10)':<30s}  {uc_val:10.4f}")
print(f"   {'FiniteClass (|H|=20)':<30s}  {gf_val:10.4f}")
print(f"   {'VC bound (d=3)':<30s}  {vc_val:10.4f}")

# 2. Sample complexity table
print("")
print("2. PAC sample complexity: n_required(eps, delta=0.05, d)")
print(f"   {'eps':>6s}  {'d=1':>8s}  {'d=3':>8s}  {'d=5':>8s}  {'d=10':>8s}  {'d=20':>8s}")
print(f"   {'-'*54}")
for eps_sc in [0.20, 0.15, 0.10, 0.05]:
    row_vals = [pac_sample_complexity_vc(eps_sc, 0.05, d_sc) for d_sc in [1, 3, 5, 10, 20]]
    print(f"   {eps_sc:6.2f}  " + "  ".join(f"{v:8,d}" for v in row_vals))

# 3. Growth function: exact vs Sauer-Shelah
print("")
print("3. Growth function (d=3): exact vs Sauer-Shelah upper bound")
print(f"   {'n':>6s}  {'Exact':>10s}  {'Sauer-Shelah':>14s}  {'Ratio':>8s}")
print(f"   {'-'*44}")
for n_gf in [5, 10, 20, 50, 100]:
    gf_exact = growth_function_exact(n_gf, 3)
    gf_poly  = growth_function_poly(n_gf, 3)
    ratio_gf = gf_poly / gf_exact if gf_exact > 0 else float("nan")
    print(f"   {n_gf:6d}  {gf_exact:10.0f}  {gf_poly:14.0f}  {ratio_gf:8.2f}")

# 4. VC vs Rademacher bound comparison
print("")
print("4. VC vs Rademacher bound comparison (d=3, delta=0.05)")
print(f"   {'n':>6s}  {'VC bound':>10s}  {'Radem est':>10s}  {'Radem bound':>12s}")
print(f"   {'-'*44}")
rng_sum = np.random.default_rng(SEED + 99)
for n_cmp in [100, 300, 500, 1000, 2000]:
    vc_b     = vc_generalisation_bound(n_cmp, 0.05, 3)
    X_cmp    = rng_sum.standard_normal((n_cmp, 2))
    clf_sum  = LogisticRegression(C=1.0, max_iter=200, random_state=SEED)
    clf_sum.fit(X_cmp, (X_cmp[:, 0] > 0).astype(int))
    pred_fn_sum = lambda Xq, _c=clf_sum: 2.0 * _c.predict(Xq).astype(float) - 1.0
    rad_est  = empirical_rademacher_complexity(X_cmp, pred_fn_sum, 200, SEED)
    rad_b    = rademacher_generalisation_bound(n_cmp, 0.05, rad_est)
    print(f"   {n_cmp:6d}  {vc_b:10.4f}  {rad_est:10.6f}  {rad_b:12.4f}")

print("")
print("=" * 70)
print("Theory-to-practice: VC and Rademacher bounds tighten as n grows,")
print("but both remain conservative vs observed generalisation gaps.")
print("PAC-Bayes and data-dependent bounds (Part 4) close this gap.")
print("=" * 70)
